In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# BlindDetection-V1 N_dev=256 calibration handoff

Prepared only. Run this detached-exact notebook once under a separate execution authorization after its reviewed producer has been pushed. The fixed denominator is 256 and the science denominator remains zero.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import torch

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
PRODUCER_EXACT = '6b08bc8cff5158c8cfced7668249d01e9892e043'
CHECKOUT = Path('/content/cegwm-blind-detection-v1-calibration')
INPUT_ROOT = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/calibration-input')
ROSTER = INPUT_ROOT / 'development-roster-256.json'
KEY_FILE = INPUT_ROOT / 'detection-key.bin'
CONFIG_FILE = INPUT_ROOT / 'calibration-config.json'
RUNS_ROOT = Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V1/calibration-runs')
RUN_ID = f"{PRODUCER_EXACT}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
RUN_ROOT = RUNS_ROOT / RUN_ID
LOCAL_ROOT = Path('/content') / (RUN_ID + '-local')
MANIFEST = RUN_ROOT / 'artifact_manifest.json'
INPUT_HASHES = {}
if RUN_ROOT.exists():
    raise FileExistsError('create-only run root required')
RUN_ROOT.mkdir(parents=True, exist_ok=False)
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
def retained_drive_artifacts():
    names = ('calibration_result.json', 'runner.stdout.txt', 'runner.stderr.txt')
    return {
        name: {'sha256': sha256(RUN_ROOT / name), 'size': (RUN_ROOT / name).stat().st_size}
        for name in names if (RUN_ROOT / name).is_file()
    }
def retain_manifest(
    stage, error, *, runner_rc=None, runner_status=None, threshold_sha256=None, status=None,
):
    if MANIFEST.exists():
        return
    payload = {
        'artifacts': retained_drive_artifacts(),
        'error': None if error is None else f'{type(error).__name__}: {error}',
        'input_hashes': dict(INPUT_HASHES),
        'producer_exact': PRODUCER_EXACT,
        'runner_rc': runner_rc,
        'runner_status': runner_status,
        'stage': stage,
        'status': status or ('CALIBRATION_COMPLETE_THRESHOLD_PENDING_LAST' if error is None else 'OPERATIONAL_BLOCKED'),
        'threshold_sha256': threshold_sha256,
    }
    with MANIFEST.open('xb') as sink:
        sink.write(json.dumps(payload, sort_keys=True, separators=(',', ':')).encode('ascii'))
stage = 'environment_guard'
try:
    if not torch.cuda.is_available():
        raise RuntimeError('GPU required; N_dev=256 was not executed')
    if CHECKOUT.exists() or LOCAL_ROOT.exists():
        raise FileExistsError('fresh checkout and local work root required')
    stage = 'detached_checkout'
    subprocess.run(['git', 'clone', REPO_URL, str(CHECKOUT)], check=True)
    subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', PRODUCER_EXACT], check=True)
    def git(*args):
        return subprocess.run(
            ['git', '-C', str(CHECKOUT), *args], check=True, capture_output=True, text=True,
        ).stdout.strip()
    if git('rev-parse', 'HEAD') != PRODUCER_EXACT or git('branch', '--show-current') != '':
        raise RuntimeError('detached producer exact differs')
    if git('status', '--porcelain=v1') != '':
        raise RuntimeError('producer checkout must be clean')
    stage = 'dependency_install'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(CHECKOUT)], check=True)
    if git('rev-parse', 'HEAD') != PRODUCER_EXACT or git('status', '--porcelain=v1') != '':
        raise RuntimeError('producer exact or clean state changed during installation')
    LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
except BaseException as error:
    retain_manifest(stage, error)
    raise


In [ ]:
stage = 'input_validation'
try:
    if not all(path.is_file() for path in (ROSTER, KEY_FILE, CONFIG_FILE)):
        raise FileNotFoundError('fixed roster, key file, or runtime config is absent')
    INPUT_HASHES.update({
        'roster_file_sha256': sha256(ROSTER),
        'runtime_config_file_sha256': sha256(CONFIG_FILE),
    })
    sys.path.insert(0, str(CHECKOUT))
    from experiments import run_blind_detection_v1 as calibration_runner
    from cegwm.shared.keys import normalize_detection_key, public_key_digest
    roster, _, roster_file_sha256 = calibration_runner.load_roster_inputs(ROSTER)
    runtime_config, config_file_sha256 = calibration_runner.load_runtime_config(CONFIG_FILE)
    key_digest = public_key_digest(normalize_detection_key(KEY_FILE.read_bytes()))
    checkpoint = Path(runtime_config['syncseal_checkpoint'])
    if not checkpoint.is_file() or sha256(checkpoint) != runtime_config['syncseal_checkpoint_sha256']:
        raise RuntimeError('SyncSeal checkpoint identity differs')
    INPUT_HASHES.update({
        'key_public_digest': key_digest,
        'syncseal_checkpoint_sha256': runtime_config['syncseal_checkpoint_sha256'],
    })
    INPUT_SUMMARY = {
        'config_file_sha256': config_file_sha256,
        'denominator': len(roster.units),
        'key_public_digest': key_digest,
        'producer_exact': PRODUCER_EXACT,
        'roster_digest': roster.digest,
        'roster_file_sha256': roster_file_sha256,
        'syncseal_checkpoint_sha256': runtime_config['syncseal_checkpoint_sha256'],
    }
    print('CEGWM_BLIND_CALIBRATION_INPUTS ' + json.dumps(INPUT_SUMMARY, sort_keys=True))
    if 'CALIBRATION_RUNNER_CALLS' not in globals():
        CALIBRATION_RUNNER_CALLS = 0
except BaseException as error:
    retain_manifest(stage, error)
    raise


In [ ]:
from google.colab import userdata
assert CALIBRATION_RUNNER_CALLS == 0
LOCAL_RESULT = LOCAL_ROOT / 'calibration_result.json'
LOCAL_CANDIDATE = LOCAL_ROOT / 'blind_detection_v1_thresholds.candidate.json'
LOCAL_STDOUT = LOCAL_ROOT / 'runner.stdout.txt'
LOCAL_STDERR = LOCAL_ROOT / 'runner.stderr.txt'
DRIVE_RESULT = RUN_ROOT / 'calibration_result.json'
DRIVE_STDOUT = RUN_ROOT / 'runner.stdout.txt'
DRIVE_STDERR = RUN_ROOT / 'runner.stderr.txt'
DRIVE_THRESHOLD = RUN_ROOT / 'blind_detection_v1_thresholds.json'
def publish_create_only(source, destination):
    with destination.open('xb') as sink:
        sink.write(source.read_bytes())
stage = 'formal_runner'
completed = None
hf_token = ''
runner_env = None
try:
    hf_token = userdata.get('HF_TOKEN')
    if not isinstance(hf_token, str) or not hf_token.strip():
        raise RuntimeError('HF_TOKEN Colab Secret is required')
    markers = ('TOKEN', 'KEY', 'SECRET', 'PASSWORD', 'CREDENTIAL')
    runner_env = {
        name: value for name, value in os.environ.items()
        if not any(marker in name.upper() for marker in markers)
    }
    runner_env['HF_TOKEN'] = hf_token
    hf_token = ''
    command = [
        sys.executable, str(CHECKOUT / 'experiments/run_blind_detection_v1.py'),
        'calibrate-and-freeze', '--roster', str(ROSTER), '--key-file', str(KEY_FILE),
        '--runtime-config', str(CONFIG_FILE), '--producer-exact', PRODUCER_EXACT,
        '--candidate-output', str(LOCAL_CANDIDATE), '--result-output', str(LOCAL_RESULT),
    ]
    CALIBRATION_RUNNER_CALLS += 1
    with LOCAL_STDOUT.open('xb') as stdout, LOCAL_STDERR.open('xb') as stderr:
        completed = subprocess.run(
            command, cwd=CHECKOUT, env=runner_env, stdout=stdout, stderr=stderr, check=False,
        )
    runner_env.pop('HF_TOKEN', None)
    runner_env = None
    if CALIBRATION_RUNNER_CALLS != 1 or not LOCAL_RESULT.is_file():
        raise RuntimeError('formal runner result is absent')
    result = json.loads(LOCAL_RESULT.read_text(encoding='ascii'))
    runner_success = (
        completed.returncode == 0
        and result.get('status') == 'CALIBRATION_COMPLETE_THRESHOLD_CANDIDATE_READY'
        and result.get('fresh_replay_zero_of_256') is True
        and result.get('threshold_candidate_ready') is True
        and LOCAL_CANDIDATE.is_file()
        and sha256(LOCAL_CANDIDATE) == result.get('threshold_candidate_sha256')
    )
    if not runner_success and LOCAL_CANDIDATE.exists():
        raise RuntimeError('threshold candidate exists for a blocked calibration')
    stage = 'result_stream_publication'
    publish_create_only(LOCAL_RESULT, DRIVE_RESULT)
    publish_create_only(LOCAL_STDOUT, DRIVE_STDOUT)
    publish_create_only(LOCAL_STDERR, DRIVE_STDERR)
    if not runner_success:
        retain_manifest(
            'terminal', RuntimeError('calibration blocked'),
            runner_rc=completed.returncode, runner_status=result.get('status'),
            status=result.get('status'),
        )
        raise RuntimeError('calibration blocked; inspect retained result and streams')
    stage = 'manifest_publication'
    retain_manifest(
        'terminal', None, runner_rc=completed.returncode,
        runner_status=result.get('status'), threshold_sha256=sha256(LOCAL_CANDIDATE),
        status='CALIBRATION_COMPLETE_THRESHOLD_PENDING_LAST',
    )
    stage = 'threshold_publication_last'
    terminal_manifest = json.loads(MANIFEST.read_text(encoding='ascii'))
    expected_threshold_sha256 = result['threshold_candidate_sha256']
    if terminal_manifest.get('threshold_sha256') != expected_threshold_sha256:
        raise RuntimeError('terminal manifest threshold identity differs')
    with DRIVE_THRESHOLD.open('xb') as sink:
        sink.write(LOCAL_CANDIDATE.read_bytes())
        sink.flush()
        os.fsync(sink.fileno())
    if sha256(DRIVE_THRESHOLD) != expected_threshold_sha256:
        raise RuntimeError('final threshold readback differs')
except BaseException as error:
    retain_manifest(
        stage, error, runner_rc=None if completed is None else completed.returncode,
        runner_status=None if not LOCAL_RESULT.is_file() else json.loads(LOCAL_RESULT.read_text()).get('status'),
    )
    raise
finally:
    hf_token = ''
    if runner_env is not None:
        runner_env.pop('HF_TOKEN', None)
    runner_env = None


In [ ]:
stored_manifest = json.loads(MANIFEST.read_text(encoding='ascii'))
if stored_manifest.get('producer_exact') != PRODUCER_EXACT:
    raise RuntimeError('stored producer exact differs')
for name, record in stored_manifest['artifacts'].items():
    artifact = RUN_ROOT / name
    if not artifact.is_file() or sha256(artifact) != record['sha256']:
        raise RuntimeError('stored artifact readback differs')
if stored_manifest.get('status') == 'CALIBRATION_COMPLETE_THRESHOLD_PENDING_LAST':
    if not DRIVE_THRESHOLD.is_file() or sha256(DRIVE_THRESHOLD) != stored_manifest['threshold_sha256']:
        raise RuntimeError('success-only threshold readback differs')
print('CEGWM_BLIND_CALIBRATION_READBACK ' + json.dumps({
    'artifact_manifest_sha256': sha256(MANIFEST),
    'producer_exact': PRODUCER_EXACT,
    'runner_status': stored_manifest['runner_status'],
}, sort_keys=True))
